In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import truncnorm
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility 
np.random.seed(42)
N_SAMPLES = 1000

print(f"Initializing Data Engine: Generating {N_SAMPLES} observations...")

# =================================================================
# STEP 1: HELPER FUNCTIONS (The "Fuzzy" Logic)
# =================================================================

def get_truncated_normal(mean=0, sd=1, low=0, upp=10):
    """
    Generates a normal distribution capped at specific boundaries.
    Useful for 'Membership Functions' like Literacy (0.0 to 1.0).
    """
    return truncnorm((low - mean) / sd, (upp - mean) / sd, loc=mean, scale=sd)

# =================================================================
# STEP 2: GENERATING PREDICTORS (The "Inputs")
# ===================================================

# 1. Client Technical Literacy (0.0 = Novice, 1.0 = Expert)
# We use a broad normal distribution centered at 0.5
literacy_gen = get_truncated_normal(mean=0.5, sd=0.25, low=0, upp=1)
client_tech_literacy = literacy_gen.rvs(N_SAMPLES)

# 2. Onboarding Owner Assigned (Boolean)
# 60% chance of being True (reflecting a typical startup)
owner_assigned = np.random.choice([0, 1], size=N_SAMPLES, p=[0.4, 0.6])

# 3. Asset Depth (0.0 = Sparse, 1.0 = Comprehensive)
# This is influenced by Owner Assignment (Accountability leads to better docs)
asset_depth = []
for owner in owner_assigned:
    mean_depth = 0.7 if owner == 1 else 0.3
    depth = get_truncated_normal(mean=mean_depth, sd=0.2, low=0, upp=1).rvs()
    asset_depth.append(depth)
asset_depth = np.array(asset_depth)

# =================================================================
# STEP 3: THE "FUZZY" INFERENCE RULES (The "Interaction Logic")
# =================================================================

# 1. Repeat Query Rate (Information Decay)
# Logic: Lower literacy AND lower assets = Exponentially more questions.
# We add "Noise" to make it look human, not robotic.
noise = np.random.normal(0, 0.05, N_SAMPLES)
repeat_query_rate = (1 - client_tech_literacy) * (1 - asset_depth) + noise
repeat_query_rate = np.clip(repeat_query_rate, 0, 1)

# 2. Founder Involvement (The "Founder Tax")
# Logic: High probability if owner is NOT assigned.
founder_involved = []
for owner in owner_assigned:
    prob = 0.85 if owner == 0 else 0.15 # 15% 'meddling' factor even with owner
    founder_involved.append(np.random.choice([0, 1], p=[1-prob, prob]))
founder_involved = np.array(founder_involved)

# 3. Time to First Value (TTFV) - The Activation Metric
# Base TTFV is 20 days. 
# Penalties: Low Literacy (+10), Sparse Assets (+15), Founder Bottleneck (+5)
base_ttfv = 20
ttfv_days = (
    base_ttfv + 
    (1 - client_tech_literacy) * 20 + 
    (1 - asset_depth) * 25 + 
    (founder_involved * 10) + 
    np.random.poisson(5, N_SAMPLES) # Natural variance
)

# 4. Satisfaction Score (1.0 to 5.0)
# Logic: Satisfaction = (Inversed Friction) + (Speed of Value)
# People hate asking questions more than they love speed.
cx_satisfaction = 5 - (repeat_query_rate * 3) - (ttfv_days / 60) + np.random.normal(0, 0.2, N_SAMPLES)
cx_satisfaction = np.clip(cx_satisfaction, 1, 5)

# =================================================================
# STEP 4: CONSOLIDATION & EXPORT
# =================================================================

df = pd.DataFrame({
    'client_id': [f"CLT_{i:04d}" for i in range(N_SAMPLES)],
    'tech_literacy': client_tech_literacy,
    'owner_assigned': owner_assigned,
    'asset_depth': asset_depth,
    'founder_involved': founder_involved,
    'repeat_query_rate': repeat_query_rate,
    'ttfv_days': ttfv_days.astype(int),
    'cx_satisfaction': cx_satisfaction
})

# Save the canonical dataset
df.to_csv('../data/onboarding_data_v2.csv', index=False)

print("SUCCESS: Canonical Dataset 'onboarding_data_v2.csv' generated in /data folder.")
print(df.head())